## PC Part Recommender Project by Tanmay
### Data Cleaning Part

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
#load all the data from the csv files
cpu = pd.read_csv("cpu.csv")
storage = pd.read_csv("internal-hard-drive.csv")
ram = pd.read_csv("memory.csv")
motherboard = pd.read_csv("motherboard.csv")
psu = pd.read_csv("power-supply.csv")
gpu = pd.read_csv("video-card.csv")
case = pd.read_csv("case.csv")

### CPU

In [ ]:
#load and merge cpu benchmark data for ranking
cpu_benchmark = pd.read_csv("CPU_UserBenchmarks.csv")
cpu_benchmark["name"] = cpu_benchmark['Brand'] + " " + cpu_benchmark['Model']
cpu = cpu.dropna(subset='price')
new_cpu = pd.merge(cpu, cpu_benchmark, on='name')
new_cpu = new_cpu.drop(['Type', 'Part Number', 'Model', 'URL', 'Samples'], axis=1).sort_values(by=["Rank"], ascending=True).reset_index()
new_cpu = new_cpu.dropna(subset='boost_clock').reset_index().drop(['level_0', 'index'], axis=1)

cpus_to_keep = [
    'AMD Ryzen 7 7800X3D', 'AMD Ryzen 5 7600X', 'AMD Ryzen 5 5600X',
    'AMD Ryzen 5 7600', 'Intel Core i9-14900K', 'Intel Core i7-14700K',
    'Intel Core i5-12400F', 'AMD Ryzen 5 3600', 'AMD Ryzen 7 7700X',
    'AMD Ryzen 9 7950X3D', 'AMD Ryzen 5 5600', 'AMD Ryzen 7 5800X',
    'Intel Core i5-12600KF', 'AMD Ryzen 5 5500', 'Intel Core i7-12700K',
    'AMD Ryzen 7 5700X', 'Intel Core i7-13700K', 'Intel Core i5-12600K',
    'AMD Ryzen 7 5800X3D', 'AMD Ryzen 9 7900X', 'AMD Ryzen 5 5600G',
    'AMD Ryzen 9 5900X', 'Intel Core i5-13600K', 'Intel Core i9-14900KS',
    'Intel Core i7-12700KF', 'Intel Core i3-12100F', 'Intel Core i5-13600KF',
    'AMD Ryzen 7 7700', 'Intel Core i9-13900K', 'Intel Core i7-14700KF',
    'Intel Core i5-13400F', 'AMD Ryzen 9 7900X3D', 'Intel Core i9-12900K',
    'AMD Ryzen 9 7950X', 'Intel Core i5-14600K', 'Intel Core i7-13700KF',
    'AMD Ryzen 7 5700G', 'AMD Ryzen 9 5950X', 'Intel Core i5-14600KF',
    'Intel Core i9-14900KF', 'Intel Core i5-14400F', 'AMD Ryzen 9 7900',
    'Intel Core i5-14500', 'Intel Core i5-12400', 'Intel Core i7-12700K',
    'Intel Core i3-12100', 'Intel Core i5-13400', 'Intel Core i7-11700K',
    'Intel Core i5-12500', 'Intel Core i7-13700', 'Intel Core i9-11900K',
    'Intel Core i5-10400', 'Intel Core i9-12900KS', 'Intel Core i5-11600K',
    'Intel Core i5-10600K', 'AMD Ryzen 5 5600X3D', 'Intel Core i9-10850K', 
    'AMD Ryzen 5 3600', 'AMD Ryzen 5 3600X', 'AMD Ryzen 7 3700X',
    'AMD Ryzen 9 3900X', 'AMD Ryzen 5 3400G', 'AMD Ryzen 3 3200G',
    'Intel Core i5-9400F', 'Intel Core i5-10400', 'Intel Core i5-10400F',
    'Intel Core i5-10600K', 'Intel Core i7-9700K', 'Intel Core i7-8700K',
    'Intel Core i5-8400', 'AMD Ryzen 5 2600', 'AMD Ryzen 5 2600X',
    'Intel Core i5-9600K'
]

new_cpu = new_cpu[new_cpu.name.isin(cpus_to_keep)].reset_index().drop(['index'], axis=1)

new_cpu.to_csv("new_cpu.csv", index=False)
new_cpu

### Motherboard

In [ ]:
#socket type to ensure compatability
socket = ['AM4', 'AM5', 'LGA1700', 'LGA1200', 'LGA1151']
motherboard = motherboard[motherboard['form_factor'] == 'ATX']
motherboard = motherboard.loc[motherboard.socket.isin(socket)].dropna(subset=['price', 'max_memory', 'memory_slots']).reset_index().drop(['index'], axis=1)

motherboard.to_csv("new_motherboard.csv", index=False)
motherboard

### Memory

In [ ]:
ram = ram.dropna(subset='price')
ram[['num_sticks', 'ram_size_per_stick']] = ram['modules'].str.split(',', expand=True)
ram[['DDR', 'speed']] = ram['speed'].str.split(',', expand=True)
ram['num_sticks'] = ram['num_sticks'].astype(int)
ram['ram_size_per_stick'] = ram['ram_size_per_stick'].astype(int)
ram['total_ram'] = ram['num_sticks'] * ram['ram_size_per_stick']
ram = ram.where(ram['total_ram'] >= 8).dropna().reset_index().drop(['index'], axis=1)
ram = ram.drop(['price_per_gb', 'color', 'modules'], axis=1)
ram = ram[ram.total_ram <= 64].reset_index().drop(['index'], axis=1)
ram['DDR'] = ram.DDR.astype(int)
ram = ram[ram.DDR >= 4].reset_index().drop(['index'], axis=1)
ram.to_csv("new_ram.csv", index=False)
ram

### Storage

In [ ]:
storage = storage.dropna(subset=['price', 'capacity']).reset_index().drop(['index'], axis=1)
storage['capacity_2'] = [f'{cap/1000} TB' if cap >= 1000 else f'{cap} GB' for cap in storage['capacity']]
uncommon_storage = [
    '16.0 TB', '24.0 TB', '8.0 TB', '1.2 TB', '2.048 TB', '12.0 TB',
    '18.0 TB', '6.0 TB', '22.0 TB', '10.0 TB', '20.0 TB', '1.6 TB',
    '1.02 TB', '5.0 TB', '4.096 TB', '14.0 TB', '7.68 TB', '1.92 TB',
    '1.05 TB', '3.84 TB', '1.5 TB', '1.8 TB', '1.9 TB', '2.4 TB',
    '3.2 TB', '3.8 TB', '32.0 GB', '40.0 GB', '64.0 GB', '80.0 GB',
    '100.0 GB', '118.0 GB', '146.0 GB', '150.0 GB', '180.0 GB',
    '200.0 GB', '275.0 GB', '300.0 GB', '320.0 GB', '360.0 GB',
    '400.0 GB', '460.0 GB', '600.0 GB', '640.0 GB', '750.0 GB',
    '800.0 GB', '900.0 GB', '1.024 TB','2.05 TB','60.0 GB','160.0 GB'
]
supported_storage = [
    'M.2 PCIe 4.0 X4',
    'M.2 PCIe 3.0 X4',
    'SATA 6.0 Gb/s',
    'M.2 SATA',
    'PCIe x8',
    'PCIe x4',
    'M.2 PCIe 3.0 X2',
]

storage = storage[~storage['capacity_2'].isin(uncommon_storage)]
storage = storage[storage.type == 'SSD']
storage = storage[storage['interface'].isin(supported_storage)].reset_index().drop(['index'], axis=1)

storage.to_csv("new_storage.csv", index=False)
storage

### GPU

In [ ]:
#gpu filtering
gpu = gpu.drop(['color', 'Type', 'Part Number', 'Samples', 'URL'], axis=1)
a = gpu.dropna().reset_index().drop(['index'], axis=1)

#removing obsolete / special use case models
outdated_nvidia = ['GTX 10', 'GTX 9', 'GTX 7', 'GTX 6', 'GT']
outdated_amd = ['RX 5', 'R9', 'R7', 'HD']
outdated_gpus = ['RX 480','RX Vega-56', 'RX Vega-64','RX 470', 'RX 460']
professional = ['Quadro RTX 8000','Titan RTX','Quadro RTX 5000','Quadro RTX 4000','Quadro RTX 6000','Quadro P4000','Quadro P2000','Quadro P1000','Titan Xp','Titan V','Quadro M2000','Quadro P5000','Quadro K600']
outdated_other = ['GeForce 8400 GS','Quadro M6000 24GB','FirePro 2270','Quadro M4000','Quadro K4200','Radeon-VII','Quadro FX 1800','Vega Frontier Edition','Quadro P6000','Quadro K2200','Quadro K620','Quadro K5200','Quadro K1200','Quadro K4000','4 Series Internal Chipset','Quadro 2000','Quadro 4000','Quadro K6000','NVS 510','GeForce 210','Quadro K2000','Quadro NVS 450','FirePro W9000','Quadro M6000','FirePro W7000','Quadro 5000','Quadro 6000','Quadro GP100','Radeon R5','Quadro NVS 295','Quadro K5000','Quadro 400','FirePro V4900','GeForce MX350','Quadro K4000M']
outdated_keywords = outdated_nvidia + outdated_amd + outdated_other + professional + outdated_gpus
filtered_df = a[~a['Model'].str.contains('|'.join(outdated_keywords), case=False, na=False)]
filtered_df = filtered_df.dropna().reset_index().drop(['index'], axis=1)

#more gpus with obsolete/special chipsets
gpus_to_drop = [
    'RTX 6000 Ada Generation',
    'RTX A6000',
    'RTX A5500',
    'RTX A5000',
    'RTX A4500',
    'RTX A4000',
    'RTX A2000 12GB',
    'RTX A2000 6GB',
    'RTX 4000 SFF Ada Generation',
    'T1000 8GB',
    'Radeon Pro WX 3100',
    'Radeon Pro WX 4100',
    'Radeon Pro WX 7100',
    'Radeon Pro WX 3200',
    'Radeon Pro VII',
    'Radeon Pro WX 5100',
    'Radeon Pro WX 2100',
    'Radeon Pro WX 9100',
    'Radeon Pro WX 8200',
    'Radeon PRO W7900',
    'Radeon PRO W7800',
    'Radeon PRO W7600',
    'Radeon PRO W7500',
    'Radeon PRO W6800',
    'Radeon PRO W6600',
    'Radeon PRO W6400',
    'Radeon Pro W5500',
    'Radeon Pro W5700',
    'Arc A380',
    'Arc A310'
]
filtered_df = filtered_df[~filtered_df['chipset'].isin(gpus_to_drop)]

power_consumption = {
    'RTX 3060': 170,
    'RTX 4070': 200,
    'RX 6700-XT': 230,
    'RTX 4060': 115,
    'RTX 4090': 450,
    'RTX 4070-Ti': 285,
    'RX 7800-XT': 270,
    'RX 7900-XTX': 355,
    'RX 6800': 250,
    'RX 7900-XT': 300,
    'RX 6600': 132,
    'RTX 3070': 220,
    'RTX 4080': 320,
    'RX 6750-XT': 250,
    'RTX 3050': 130,
    'RTX 3060-Ti': 200,
    'RTX 4060-Ti': 160,
    'RX 7700-XT': 245,
    'RX 7600': 165,
    'RX 6650-XT': 180,
    'RX 6800-XT': 300,
    'RTX 3080': 320,
    'RX 6600-XT': 160,
    'RTX 2060': 160,
    'Arc A770': 225,
    'RX 6500-XT': 107,
    'RTX 3080-Ti': 350,
    'RX 6950-XT': 335,
    'RTX 3070-Ti': 290,
    'RTX 3090': 350,
    'RTX 2070': 175,
    'RTX 3090-Ti': 450,
    'RTX 2080-Ti': 250,
    'Arc A580': 165,
    'RTX 2080': 215,
    'RX 6900-XT': 300,
    'Gigabyte RTX 3060 Ti 8GB Gaming OC Pro': 200,
    'Arc A380': 75,
    'Arc A750': 225,
    'Asus RTX 3070 Ti 8GB Tuf OC': 290 
}

filtered_df = filtered_df.drop(['best_match_model', 'merge_key'], axis=1)
filtered_df['tdp'] = filtered_df['Model'].map(power_consumption)
filtered_df = filtered_df.dropna(subset=['tdp', 'Rank']).reset_index().drop(['index',], axis=1)
filtered_df.to_csv("new_gpu.csv", index=False)
filtered_df

### Power Supply

In [ ]:
psu = psu.dropna().reset_index().drop(['index'], axis=1)
wattages_to_keep = [550, 650, 750, 850, 1000, 1200]
psu = psu[psu['wattage'].isin(wattages_to_keep)]
psu = psu[psu.type == 'ATX'].reset_index().drop(['index'], axis=1)
psu.to_csv("new_psu.csv", index=False)
psu

### Case

In [ ]:
case = case.dropna(subset='price').reset_index().drop(['index'], axis=1)
case = case.drop(['psu', 'external_volume'], axis=1)
cases_to_keep = ['ATX Mid Tower', 'ATX Full Tower']
case_max_gpu_length = {
    'ATX Mid Tower': 330,
    'ATX Full Tower': 400,
}
case = case[case['type'].isin(cases_to_keep)]
case['max_gpu_length'] = case['type'].map(case_max_gpu_length).reset_index().drop(['index'], axis=1)
case.to_csv("new_case.csv", index=False)
case

## Adding additional features to help with model and comptability

In [ ]:
#loading the cleaned datasets for gpu 
gpus = pd.read_csv("new_gpu.csv")
#adding a column for price/performance ratio using core clock data
cc2p = gpus.core_clock / gpus.price
bc2p = gpus.boost_clock / gpus.price
gpus['avg_p2p'] = (cc2p + bc2p)/2    #this is the average price-to-performance calculation using the clock speeds and prices
gpus.to_csv("new_gpu_with_features.csv", index=False)
gpus
#gpus[gpus.price <= 567.996].sort_values(by=['Rank', 'price', 'avg_p2p'], ascending=[True, True, False])   filter test

In [ ]:
#same for cpu
cpus = pd.read_csv("new_cpu.csv")
cc2p = cpus.core_clock / cpus.price
bc2p = cpus.boost_clock / cpus.price
cpus['avg_p2p'] = ((cc2p+bc2p)/2)*100

def get_socket(cpu_name):
    if any(x in cpu_name for x in ['Ryzen 9 7', 'Ryzen 7 7', 'Ryzen 5 7']):
        return 'AM5'
    if 'Ryzen' in cpu_name:
        return 'AM4'
    if 'Core' in cpu_name:
        parts = cpu_name.split('-')
        if len(parts) > 1:
            gen_part = parts[1].split('K')[0] 
            if len(gen_part) > 1 and gen_part[0] == '1':
                gen = int(gen_part[:2]) 
            else:
                gen = int(gen_part[0]) 
            if gen >= 12:
                return 'LGA1700'
            if gen in [10, 11]:
                return 'LGA1200'
            return 'LGA1151'

cpus['socket'] = cpus['name'].apply(get_socket)

cpus.to_csv("new_cpu_with_features.csv", index=False)
cpus
cpus[cpus.price <= 500].sort_values(by=['Rank', 'price', 'avg_p2p'], ascending=[True, True, False]) 

In [ ]:
rams = pd.read_csv("new_ram.csv")
rams['speed_to_price'] = rams.speed / rams.price
rams['total_ram_to_price'] = rams.total_ram / rams.price
rams.to_csv("new_ram_with_features.csv", index=False)
rams